In [ ]:
import os
import gc
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import matplotlib as mpl

import anndata as ad
import numpy as np
import yaml
import sys
import h5py
import logging
import scanpy as sc
from os.path import join
import scipy.io as sio
import scipy.sparse as sps
from sklearn.cluster import KMeans
import gzip
from scipy.io import mmread
from pathlib import Path, PurePath
from sklearn.metrics import adjusted_rand_score
from annoy import AnnoyIndex
import itertools
from sklearn.preprocessing import normalize

import metrics
from spamosaic.preprocessing import Epigenome_preprocess

# MB_RNA+ATAC

In [ ]:
def reorder(ad1, ad2):
    shared_barcodes = ad1.obs_names.intersection(ad2.obs_names)
    ad1 = ad1[shared_barcodes].copy()
    ad2 = ad2[shared_barcodes].copy()
    return ad1, ad2

def load_peak_expr(_dir):
    data = sio.mmread(join(_dir, 'data.mtx'))
    cname = pd.read_csv(join(_dir, 'barcode.csv'), index_col=0)['x'].to_list()
    feat = pd.read_csv(join(_dir, 'feat.csv'), index_col=0)['x'].to_list()
    ad = sc.AnnData(sps.csr_matrix(data.T))
    ad.obs_names = cname
    ad.var_names = feat
    return ad

In [ ]:
data_dir = '../../data/raw/MB-5M/mouse_brain_rna+atac'

df1_rna = pd.read_csv(join(data_dir, 'rna+atac/GSM6204636_MouseBrain_20um_matrix.tsv'), sep='\t')
df1_spatial_pos = pd.read_csv(join(data_dir, 'rna+atac/GSM6204623_MouseBrain_20um_spatial_rna_part/tissue_positions_list.csv'), header=None, index_col=0)
ad1_rna = sc.AnnData(df1_rna.T, obsm={'spatial': df1_spatial_pos.loc[df1_rna.columns, [2, 3]].values})

ad1_atac = load_peak_expr(join(data_dir, 'rna+atac/For_Imputation_Task/GSM6204623_peak_data'))
df1_atac_spatial = pd.read_csv(join(data_dir, 'rna+atac/GSM6204623_MouseBrain_20um_spatial_rna_part/tissue_positions_list.csv'), index_col=0, header=None)
ad1_atac.obsm['spatial'] = df1_atac_spatial.loc[ad1_atac.obs_names, [2, 3]].values
ad1_rna, ad1_atac = reorder(ad1_rna, ad1_atac)

# ===
df2_rna = pd.read_csv(join(data_dir, 'rna+atac/GSM6753041_MouseBrain_20um_repATAC_matrix.tsv'), sep='\t')
df2_rna_spatial = pd.read_csv(join(data_dir, 'rna+atac/GSM6753041_MouseBrain_20um_repATAC_spatial/tissue_positions_list.csv'), index_col=0, header=None)
ad2_rna = sc.AnnData(df2_rna.T, obsm={'spatial': df2_rna_spatial.loc[df2_rna.columns, [2, 3]].values})

ad2_atac = load_peak_expr(join(data_dir, 'rna+atac/For_Imputation_Task/GSM6758284_peak_data'))
df2_atac_spatial = pd.read_csv(join(data_dir, 'rna+atac/GSM6753041_MouseBrain_20um_repATAC_spatial/tissue_positions_list.csv'), index_col=0, header=None)
ad2_atac.obsm['spatial'] = df2_atac_spatial.loc[ad2_atac.obs_names, [2, 3]].values
ad2_rna, ad2_atac = reorder(ad2_rna, ad2_atac)

# ===
df3_rna = pd.read_csv(join(data_dir, 'rna+atac/GSM6753043_MouseBrain_20um_100barcodes_ATAC_matrix.tsv'), sep='\t')
df3_rna_spatial = pd.read_csv(join(data_dir, 'rna+atac/GSM6753043_MouseBrain_20um_100barcodes_ATAC_spatial/tissue_positions_list.csv'), index_col=0, header=None)
ad3_rna = sc.AnnData(df3_rna.T, obsm={'spatial': df3_rna_spatial.loc[df3_rna.columns, [2, 3]].values})

ad3_atac = load_peak_expr(join(data_dir, 'rna+atac/For_Imputation_Task/GSM6758285_peak_data'))
df3_atac_spatial = pd.read_csv(join(data_dir, 'rna+atac//GSM6753043_MouseBrain_20um_100barcodes_ATAC_spatial/tissue_positions_list.csv'), index_col=0, header=None)
ad3_atac.obsm['spatial'] = df3_atac_spatial.loc[ad3_atac.obs_names, [2, 3]].values
ad3_rna, ad3_atac = reorder(ad3_rna, ad3_atac)

shared_gene = ad1_rna.var_names.intersection(ad2_rna.var_names).intersection(ad3_rna.var_names)
shared_peak = ad1_atac.var_names.intersection(ad2_atac.var_names).intersection(ad3_atac.var_names)
ad1_rna = ad1_rna[:, shared_gene].copy(); ad2_rna = ad2_rna[:, shared_gene].copy(); ad3_rna = ad3_rna[:, shared_gene].copy()
ad1_atac = ad1_atac[:, shared_peak].copy(); ad2_atac = ad2_atac[:, shared_peak].copy(); ad3_atac = ad3_atac[:, shared_peak].copy()

ad1_rna.obs_names = [f's1-{_}' for _ in ad1_rna.obs_names]
ad1_atac.obs_names = [f's1-{_}' for _ in ad1_atac.obs_names]
ad2_rna.obs_names = [f's2-{_}' for _ in ad2_rna.obs_names]
ad2_atac.obs_names = [f's2-{_}' for _ in ad2_atac.obs_names]
ad3_rna.obs_names = [f's3-{_}' for _ in ad3_rna.obs_names]
ad3_atac.obs_names = [f's3-{_}' for _ in ad3_atac.obs_names]

ad1_rna.obs['src'] = ['s1']*ad1_rna.n_obs
ad1_atac.obs['src'] = ['s1']*ad1_atac.n_obs
ad2_rna.obs['src'] = ['s2']*ad2_rna.n_obs
ad2_atac.obs['src'] = ['s2']*ad2_atac.n_obs
ad3_rna.obs['src'] = ['s3']*ad3_rna.n_obs
ad3_atac.obs['src'] = ['s3']*ad3_atac.n_obs

In [ ]:
ad_rna_all = sc.concat([ad1_rna, ad2_rna, ad3_rna])
ad_atac_all = sc.concat([ad1_atac, ad2_atac, ad3_atac])

sc.pp.highly_variable_genes(ad_rna_all, flavor='seurat_v3', n_top_genes=5000, batch_key='src')
hvg_names = ad_rna_all.var.query('highly_variable').index.to_numpy()

# ac.pp.tfidf(ad_atac_all, scale_factor=1e4)
sc.pp.highly_variable_genes(ad_atac_all, flavor='seurat_v3', n_top_genes=50000, batch_key='src')
hvp_names = ad_atac_all.var.query('highly_variable').index.to_numpy()

In [ ]:
ad1_rna = ad1_rna[:, hvg_names].copy(); ad1_atac = ad1_atac[:, hvp_names].copy()
ad2_rna = ad2_rna[:, hvg_names].copy(); ad2_atac = ad2_atac[:, hvp_names].copy()
ad3_rna = ad3_rna[:, hvg_names].copy(); ad3_atac = ad3_atac[:, hvp_names].copy()

In [ ]:
## filter feat names
filtered_atac_feats = [_ for _ in ad1_atac.var_names if _.startswith('chr')]
ad1_atac = ad1_atac[:, filtered_atac_feats].copy()
ad2_atac = ad2_atac[:, filtered_atac_feats].copy()
ad3_atac = ad3_atac[:, filtered_atac_feats].copy()

In [ ]:
RNA_ADS = [ad1_rna, ad2_rna, ad3_rna]
ATAC_ADS = [ad1_atac, ad2_atac, ad3_atac]
mod_dict = {'rna': RNA_ADS, 'atac':ATAC_ADS}

In [ ]:
out_dir = '../../results/imputations'
methods = ['SpaMosaic', 'MultiVI', 'BABEL', 'MIDAS']

cv_res = []
for mi in methods:
    print(mi)
    for i in range(3):
        gt_ad = ATAC_ADS[i]
        gt_X  = gt_ad.X.A if sps.issparse(gt_ad.X) else gt_ad.X
        pr_ad = sc.read_h5ad(join(out_dir, mi, f'MB_imputation/cv{i}_imputedATAC.h5ad'))
        pr_X  = pr_ad.X.A if sps.issparse(pr_ad.X) else pr_ad.X
        auc = metrics.eval_AUC_all(gt_X, pr_X, bin_thr=1)
        pcc_cell, pcc_peak = metrics.PCCs(gt_X, pr_X)
        pcc_cell, pcc_peak = np.nanmean(pcc_cell), np.nanmean(pcc_peak)
        cmd_cell = metrics.CMD(pr_X, gt_X)
        cmd_peak = metrics.CMD(pr_X.T, gt_X.T)
        cv_res.append([mi, i, auc, pcc_cell, pcc_peak, cmd_cell, cmd_peak])

In [ ]:
score_dir = '../../results/scores/MB_imputation'
os.makedirs(score_dir, exist_ok=True)

df_cv_res = pd.DataFrame(cv_res, columns=['method', 'cv-fold', 'smthed_auroc', 'smthed_PCC_cell', 'smthed_PCC_peak', 
                                          'smthed_cmd_cell', 'smthed_cmd_peak'])
df_cv_res.to_csv(f'{score_dir}/df_imputation.csv')

In [ ]:
out_dir = '../../results/imputations'
methods = ['SpaMosaic', 'MultiVI', 'BABEL', 'MIDAS']

cv_res = []
for mi in methods:
    print(mi)
    for i in range(3):
        gt_ad = ATAC_ADS[i]
        Epigenome_preprocess([gt_ad], batch_corr=False, n_peak=50000, batch_key='src', key='X_lsi', return_hvf=False)
        pr_ad = sc.read_h5ad(join(out_dir, mi, f'MB_imputation/cv{i}_imputedATAC.h5ad'))

        smth_gt_X = metrics.knn_smoothing(gt_ad, dim_red_key='X_lsi', knn=50)
        # gt_X  = gt_ad.X.A if sps.issparse(gt_ad.X) else gt_ad.X
        pr_X  = pr_ad.X.A if sps.issparse(pr_ad.X) else pr_ad.X

        auc = metrics.eval_AUC_all(smth_gt_X, pr_X, bin_thr=1)
        
        pcc_cell, pcc_peak = metrics.PCCs(smth_gt_X, pr_X)
        pcc_cell, pcc_peak = np.nanmean(pcc_cell), np.nanmean(pcc_peak)

        cmd_cell = metrics.CMD(pr_X, smth_gt_X)
        cmd_peak = metrics.CMD(pr_X.T, smth_gt_X.T)

        # cv_res.append([mi, i, pcc_cell, pcc_peak])
        cv_res.append([mi, i, auc, pcc_cell, pcc_peak, cmd_cell, cmd_peak])

In [ ]:
score_dir = '../../results/scores/MB_imputation'
os.makedirs(score_dir, exist_ok=True)

df_cv_res = pd.DataFrame(cv_res, columns=['method', 'cv-fold', 'smthed_auroc', 'smthed_PCC_cell', 'smthed_PCC_peak', 
                                          'smthed_cmd_cell', 'smthed_cmd_peak'])
df_cv_res.to_csv(f'{score_dir}/df_imputation_smthed.csv')

# MISAR

In [ ]:
def subset_ad(ad, subset_index):
    ad = ad[subset_index].copy()
    return ad

def load_peak_expr(_dir):
    data = sio.mmread(join(_dir, 'data.mtx'))
    cname = pd.read_csv(join(_dir, 'barcode.csv'), index_col=0).index.to_list()
    feat = pd.read_csv(join(_dir, 'feat.csv'), index_col=0)['x'].to_list()
    ad = sc.AnnData(sps.csr_matrix(data.T))
    ad.obs_names = cname
    ad.var_names = feat
    return ad

def load_h5(path):
    with h5py.File(path, 'r') as f:
        print(f['matrix'].keys())
        print(f['matrix']['features'].keys())

        barcodes = [_.decode('utf-8') for _ in f['matrix']['barcodes'][:]]
        data = f['matrix']['data'][:]
        indices = f['matrix']['indices'][:]
        indptr = f['matrix']['indptr'][:]
        shape = f['matrix']['shape'][:]

        feature_type = [_.decode('utf-8') for _ in f['matrix']['features']['feature_type'][:]]
        feature_id   = [_.decode('utf-8') for _ in f['matrix']['features']['id'][:]]
        feature_name = [_.decode('utf-8') for _ in f['matrix']['features']['name'][:]]
        feature_interval = [_.decode('utf-8') for _ in f['matrix']['features']['interval'][:]]


        X = sps.csc_matrix(
            (data, indices, indptr), 
            shape = shape
        ).tocsc().astype(np.float32).T.toarray()

        adata = sc.AnnData(X)
        adata.obs_names = barcodes
        adata.var_names = feature_id
        adata.var['type'] = feature_type
        adata.var['name'] = feature_name
        adata.var['interval'] = feature_interval
    return adata

In [ ]:
data_dir = '../../data/raw/MISAR_seq/'
ad_bridge = load_h5(join(data_dir, 'E15_5-S1_raw_feature_bc_matrix.h5'))
ad_test1 = load_h5(join(data_dir, 'E13_5-S1_raw_feature_bc_matrix.h5'))
ad_test2 = load_h5(join(data_dir, 'E18_5-S1_raw_feature_bc_matrix.h5'))  # inconsistent peak name across batches
peak_mat = sps.csr_matrix(sio.mmread(join(data_dir, 'BaiduDisk/section1/peak_mat.mtx')).T)
peak_spot_name = pd.read_csv(join(data_dir, 'BaiduDisk/section1/peak_spot_names.csv')).x.values

meta = pd.read_csv(join(data_dir, 'BaiduDisk/section1/meta_data.csv'), index_col=0)

ad_bridge.obs_names = [f'E15_5-S1#{_}' for _ in ad_bridge.obs_names]
ad_test1.obs_names = [f'E13_5-S1#{_}' for _ in ad_test1.obs_names]
ad_test2.obs_names = [f'E18_5-S1#{_}' for _ in ad_test2.obs_names]

# split rna and peak
ad15_rna = ad_bridge[:, ad_bridge.var['type'] == 'Gene Expression'].copy()
ad13_rna = ad_test1[:, ad_test1.var['type'] == 'Gene Expression'].copy()
ad18_rna = ad_test2[:, ad_test2.var['type'] == 'Gene Expression'].copy()

ad13_atac = load_peak_expr(join(data_dir, 'S1-E13-E15-18-peak_data/E13'))
ad15_atac = load_peak_expr(join(data_dir, 'S1-E13-E15-18-peak_data/E15'))
ad18_atac = load_peak_expr(join(data_dir, 'S1-E13-E15-18-peak_data/E18'))
ad13_atac.obs_names = [f'E13_5-S1#{_}' for _ in ad13_atac.obs_names]
ad15_atac.obs_names = [f'E15_5-S1#{_}' for _ in ad15_atac.obs_names]
ad18_atac.obs_names = [f'E18_5-S1#{_}' for _ in ad18_atac.obs_names]

ad15_rna = subset_ad(ad15_rna, ad15_rna.obs_names.intersection(meta.index))
ad13_rna = subset_ad(ad13_rna, ad13_rna.obs_names.intersection(meta.index))
ad18_rna = subset_ad(ad18_rna, ad18_rna.obs_names.intersection(meta.index))

ad15_rna.obs = meta.loc[ad15_rna.obs_names].copy()
ad15_atac.obs = meta.loc[ad15_atac.obs_names].copy()
ad13_rna.obs = meta.loc[ad13_rna.obs_names].copy()
ad13_atac.obs = meta.loc[ad13_atac.obs_names].copy()
ad18_rna.obs = meta.loc[ad18_rna.obs_names].copy()
ad18_atac.obs = meta.loc[ad18_atac.obs_names].copy()

ad15_atac = ad15_atac[ad15_rna.obs_names].copy()
ad13_atac = ad13_atac[ad13_rna.obs_names].copy()
ad18_rna  = ad18_rna[ad18_atac.obs_names].copy()  

del peak_mat, ad_bridge, ad_test1, ad_test2
gc.collect()

In [ ]:
ad13_rna.obs['src'] = ad13_atac.obs['src'] = ['e13']*ad13_rna.n_obs
ad15_rna.obs['src'] = ad15_atac.obs['src'] = ['e15']*ad15_rna.n_obs
ad18_rna.obs['src'] = ad18_atac.obs['src'] = ['e18']*ad18_rna.n_obs

In [ ]:
ad_rna_all = sc.concat([ad13_rna, ad15_rna, ad18_rna])
ad_atac_all = sc.concat([ad13_atac, ad15_atac, ad18_atac])

sc.pp.highly_variable_genes(ad_rna_all, flavor='seurat_v3', n_top_genes=5000, batch_key='src')
hvg_names = ad_rna_all.var.query('highly_variable').index.to_numpy()

sc.pp.highly_variable_genes(ad_atac_all, flavor='seurat_v3', n_top_genes=50000, batch_key='src')
hvp_names = ad_atac_all.var.query('highly_variable').index.to_numpy()

In [ ]:
ad13_rna = ad13_rna[:, hvg_names].copy(); ad13_atac = ad13_atac[:, hvp_names].copy()
ad15_rna = ad15_rna[:, hvg_names].copy(); ad15_atac = ad15_atac[:, hvp_names].copy()
ad18_rna = ad18_rna[:, hvg_names].copy(); ad18_atac = ad18_atac[:, hvp_names].copy()

## filter feat names
filtered_atac_feats = [_ for _ in ad13_atac.var_names if _.startswith('chr')]
ad13_atac = ad13_atac[:, filtered_atac_feats].copy()
ad15_atac = ad15_atac[:, filtered_atac_feats].copy()
ad18_atac = ad18_atac[:, filtered_atac_feats].copy()

In [ ]:
RNA_ADS = [ad13_rna, ad15_rna, ad18_rna]
ATAC_ADS = [ad13_atac, ad15_atac, ad18_atac]
mod_dict = {'rna': RNA_ADS, 'atac':ATAC_ADS}
n_batches = 3
mod_sets = ['rna', 'atac']

In [ ]:
out_dir = '../../results/imputations'
methods = ['SpaMosaic', 'MultiVI', 'BABEL', 'MIDAS']

cv_res = []
for mi in methods:
    print(mi)
    for i in range(3):
        gt_ad = ATAC_ADS[i]
        gt_X  = gt_ad.X.A if sps.issparse(gt_ad.X) else gt_ad.X
        pr_ad = sc.read_h5ad(join(out_dir, mi, f'Misar_imputation/cv{i}_imputedATAC.h5ad'))
        pr_X  = pr_ad.X.A if sps.issparse(pr_ad.X) else pr_ad.X
        auc = metrics.eval_AUC_all(gt_X, pr_X, bin_thr=1)
        pcc_cell, pcc_peak = metrics.PCCs(gt_X, pr_X)
        pcc_cell, pcc_peak = np.nanmean(pcc_cell), np.nanmean(pcc_peak)
        cmd_cell = metrics.CMD(pr_X, gt_X)
        cmd_peak = metrics.CMD(pr_X.T, gt_X.T)
        cv_res.append([mi, i, auc, pcc_cell, pcc_peak, cmd_cell, cmd_peak])

In [ ]:
score_dir = '../../results/scores/Misar_imputation'
os.makedirs(score_dir, exist_ok=True)

df_cv_res = pd.DataFrame(cv_res, columns=['method', 'cv-fold', 'smthed_auroc', 'smthed_PCC_cell', 'smthed_PCC_peak', 
                                          'smthed_cmd_cell', 'smthed_cmd_peak'])
df_cv_res.to_csv(f'{score_dir}/df_imputation.csv')

In [ ]:
out_dir = '../../results/imputations'
methods = ['SpaMosaic', 'MultiVI', 'BABEL', 'MIDAS']

cv_res = []
for mi in methods:
    print(mi)
    for i in range(3):
        gt_ad = ATAC_ADS[i]
        Epigenome_preprocess([gt_ad], batch_corr=False, n_peak=50000, batch_key='src', key='X_lsi', return_hvf=False)
        pr_ad = sc.read_h5ad(join(out_dir, mi, f'Misar_imputation/cv{i}_imputedATAC.h5ad'))

        smth_gt_X = metrics.knn_smoothing(gt_ad, dim_red_key='X_lsi', knn=50)
        # gt_X  = gt_ad.X.A if sps.issparse(gt_ad.X) else gt_ad.X
        pr_X  = pr_ad.X.A if sps.issparse(pr_ad.X) else pr_ad.X

        auc = metrics.eval_AUC_all(smth_gt_X, pr_X, bin_thr=1)
        
        pcc_cell, pcc_peak = metrics.PCCs(smth_gt_X, pr_X)
        pcc_cell, pcc_peak = np.nanmean(pcc_cell), np.nanmean(pcc_peak)

        cmd_cell = metrics.CMD(pr_X, smth_gt_X)
        cmd_peak = metrics.CMD(pr_X.T, smth_gt_X.T)

        # cv_res.append([mi, i, pcc_cell, pcc_peak])
        cv_res.append([mi, i, auc, pcc_cell, pcc_peak, cmd_cell, cmd_peak])

In [ ]:
score_dir = '../../results/scores/Misar_imputation'
os.makedirs(score_dir, exist_ok=True)

df_cv_res = pd.DataFrame(cv_res, columns=['method', 'cv-fold', 'smthed_auroc', 'smthed_PCC_cell', 'smthed_PCC_peak', 
                                          'smthed_cmd_cell', 'smthed_cmd_peak'])
df_cv_res.to_csv(f'{score_dir}/df_imputation_smthed.csv')

In [ ]:
def flat_df(df, flat_cols=[], k_k='task', v_k='score'):
    other_cols = list(set(df.columns) - set(flat_cols))
    dfs = []
    for col in flat_cols:
        tmp_df = df[[col] + other_cols].copy()
        tmp_df.columns = [v_k] + other_cols
        tmp_df[k_k] = col
        dfs.append(tmp_df)
    return pd.concat(dfs)

In [ ]:
df_imp_res1 = pd.read_csv('/media/asus/data16t/xuhua/gitrepo/scHulk_server/BridgeNorm/figures/imputation/3slices_MB_RNA+ATAC/res.csv', index_col=0)
df_imp_res1_supp = pd.read_csv('/media/asus/data16t/xuhua/gitrepo/scHulk_server/BridgeNorm/figures/imputation/3slices_MB_RNA+ATAC/res_smthed_res.csv', index_col=0)
# df_imp_res1 = df_imp_res1.merge(df_imp_res1_supp, on=['method', 'cv-fold'], how='inner')

In [ ]:
df_imp_res1.loc[df_imp_res1.method=='ours']

In [ ]:
df_imp_res1_supp.loc[df_imp_res1_supp.method=='ours']

In [ ]:
df_imp_res2 = pd.read_csv('/media/asus/data16t/xuhua/gitrepo/scHulk_server/BridgeNorm/figures/imputation/Misar_E13-E15-E18/res.csv', index_col=0)
df_imp_res2_supp = pd.read_csv('/media/asus/data16t/xuhua/gitrepo/scHulk_server/BridgeNorm/figures/imputation/Misar_E13-E15-E18/res_smthed_res.csv', index_col=0)
# df_imp_res2 = df_imp_res2.merge(df_imp_res2_supp, on=['method', 'cv-fold'], how='inner')

df_imp_res1['dataset'] = 'Mouse postnatal brain'
df_imp_res2['dataset'] = 'Mouse embryonic brain'
df_imp_res1_supp['dataset'] = 'Mouse postnatal brain'
df_imp_res2_supp['dataset'] = 'Mouse embryonic brain'

df_imp_res = pd.concat([df_imp_res1, df_imp_res2])
df_imp_res_supp = pd.concat([df_imp_res1_supp, df_imp_res2_supp])

df_imp_res_flat = flat_df(df_imp_res, 
                           flat_cols=['AUROC', 'PCC_cell', 'PCC_peak', 'CMD_cell', 'CMD_peak'], 
                           k_k='task', v_k='score')

df_imp_res_supp_flat = flat_df(df_imp_res_supp, 
                           flat_cols=['smthed_auroc', 'smthed_PCC_cell', 'smthed_PCC_peak', 'smthed_cmd_cell', 'smthed_cmd_peak'], 
                           k_k='task', v_k='score')

In [ ]:
method={
    'midas':'MIDAS',
    'multiVI':'MultiVI',
    'babel':'BABEL',
    'ours':'SpaMosaic'
}
taskd = {
    'AUROC':'AUROC',
    'PCC_cell':'Spot-spot PCC', 
    'PCC_peak':'Peak-peak PCC', 
    'CMD_cell':'Spot-spot CMD', 
    'CMD_peak':'Peak-peak CMD'
}
taskd2 = {
    'smthed_auroc':'AUROC',
    'smthed_PCC_cell':'Spot-spot PCC', 
    'smthed_PCC_peak':'Peak-peak PCC', 
    'smthed_cmd_cell':'Spot-spot CMD', 
    'smthed_cmd_peak':'Peak-peak CMD'
}

df_imp_res_flat['method'] = df_imp_res_flat['method'].map(method)
df_imp_res_flat['task'] = df_imp_res_flat['task'].map(taskd)

df_imp_res_supp_flat['method'] = df_imp_res_supp_flat['method'].map(method)
df_imp_res_supp_flat['task'] = df_imp_res_supp_flat['task'].map(taskd2)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

fs = 14  # 字体大小

g = sns.catplot(
    data=df_imp_res_flat, 
    x="task", y='score', hue='method', col='dataset',
    kind="bar", height=5, aspect=1., 
)

# 去掉 xlabel 和 ylabel
g.set_axis_labels("", "")

# 去掉子图标题里的 "dataset="
for ax in g.axes.flat:
    title = ax.get_title()
    new_title = title.replace("dataset = ", "").replace("dataset=", "")
    ax.set_title(new_title, fontname='Arial', fontsize=fs)

# 设置 x 和 y tick 标签字体和角度
yticks = None
for ax in g.axes.flat:
    yticks = ax.get_yticklabels() if yticks is None else yticks
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='center', fontname='Arial', fontsize=fs-2)
    ax.set_yticklabels(yticks, fontname='Arial', fontsize=fs-2)

    ax.title.set_fontname('Arial')
    ax.title.set_fontsize(fs)
    ax.xaxis.label.set_fontname('Arial')
    ax.xaxis.label.set_fontsize(fs-2)
    ax.yaxis.label.set_fontname('Arial')
    ax.yaxis.label.set_fontsize(fs-2)

# 去掉 legend title
g._legend.set_title(None)

# 设置 legend 字体，并移动到右侧中间
for text in g._legend.texts:
    text.set_fontname('Arial')
    text.set_fontsize(fs-2)

g._legend.set_bbox_to_anchor((1.12, 0.5))  # 右侧中间
g._legend.set_frame_on(False)  # 可选：去掉边框

# 调整整体布局以防 legend 被截断
plt.subplots_adjust(right=0.85)
plt.tight_layout()

g.fig.savefig(f"/media/asus/data16t/xuhua/gitrepo/scHulk_server/BridgeNorm/SpaMosaic-review/real_mosaic_case/out/imputation/mouse_brain.pdf", bbox_inches='tight')

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

fs = 14  # 字体大小

g = sns.catplot(
    data=df_imp_res_supp_flat, 
    x="task", y='score', hue='method', col='dataset',
    kind="bar", height=5, aspect=1., 
)

# 去掉 xlabel 和 ylabel
g.set_axis_labels("", "")

# 去掉子图标题里的 "dataset="
for ax in g.axes.flat:
    title = ax.get_title()
    new_title = title.replace("dataset = ", "").replace("dataset=", "")
    ax.set_title(new_title, fontname='Arial', fontsize=fs)

# 设置 x 和 y tick 标签字体和角度
yticks = None
for ax in g.axes.flat:
    yticks = ax.get_yticklabels() if yticks is None else yticks
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='center', fontname='Arial', fontsize=fs-2)
    ax.set_yticklabels(yticks, fontname='Arial', fontsize=fs-2)

    ax.title.set_fontname('Arial')
    ax.title.set_fontsize(fs)
    ax.xaxis.label.set_fontname('Arial')
    ax.xaxis.label.set_fontsize(fs-2)
    ax.yaxis.label.set_fontname('Arial')
    ax.yaxis.label.set_fontsize(fs-2)

# 去掉 legend title
g._legend.set_title(None)

# 设置 legend 字体，并移动到右侧中间
for text in g._legend.texts:
    text.set_fontname('Arial')
    text.set_fontsize(fs-2)

g._legend.set_bbox_to_anchor((1.12, 0.5))  # 右侧中间
g._legend.set_frame_on(False)  # 可选：去掉边框

# 调整整体布局以防 legend 被截断
plt.subplots_adjust(right=0.85)
plt.tight_layout()

g.fig.savefig(f"/media/asus/data16t/xuhua/gitrepo/scHulk_server/BridgeNorm/SpaMosaic-review/real_mosaic_case/out/imputation/mouse_brain_smthed.pdf", bbox_inches='tight')

In [ ]:
import scanpy as sc
from 
_dir = '/media/asus/data16t/huangzt/scNCL_data/scNCL/MCA/scjoint/data_subset'

adx = sc.read_h5ad(join(_dir, 'adata_atac_cache.h5ad'))

In [ ]:
adx